# VitalDB reduced-model Colab GPU validation

This notebook performs environment and data validation, tests, two validation-only smoke runs, and a short CUDA benchmark. It does not run full scientific training, group retraining, feature selection, RL, or test-set importance analysis. Smoke attention is pipeline diagnostics only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = DRIVE_PROJECT_ROOT / 'data/modeling/full'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs'
ENV_OUTPUT_DIR = OUTPUT_ROOT / 'colab_environment'
GRU_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/gru/seed_42'
ATTENTION_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/attention/seed_42'
BENCHMARK_DIR = OUTPUT_ROOT / 'runtime_benchmark'
for path in (ENV_OUTPUT_DIR, GRU_SMOKE_DIR.parent, ATTENTION_SMOKE_DIR.parent, BENCHMARK_DIR):
    path.mkdir(parents=True, exist_ok=True)
print({'dataset': str(DATASET_DIR), 'outputs': str(OUTPUT_ROOT)})

In [ ]:
import subprocess

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Active git commit:', commit)

## Preserve Colab CUDA PyTorch
The dependency script installs only missing non-PyTorch packages. It rejects `torch`, `torchvision`, or `torchaudio` in the Colab requirement list and verifies that the PyTorch version did not change.

In [ ]:
import os
import sys
import torch

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Before dependency check:', torch.__version__, torch.version.cuda, torch.cuda.is_available())
subprocess.run([sys.executable, 'scripts/install_colab_dependencies.py'], check=True)

In [ ]:
# This writes the audit before raising if a GPU was not assigned.
subprocess.run([
    sys.executable, 'scripts/audit_colab_environment.py',
    '--repo-dir', str(REPO_DIR),
    '--output', str(ENV_OUTPUT_DIR / 'environment_audit.json'),
], check=True)

In [ ]:
subprocess.run([
    sys.executable, 'scripts/validate_modeling_artifacts.py',
    '--dataset-dir', str(DATASET_DIR),
    '--output', str(ENV_OUTPUT_DIR / 'data_artifact_audit.json'),
], check=True)

In [ ]:
test_result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True
)
test_log = test_result.stdout + test_result.stderr
print(test_log)
(ENV_OUTPUT_DIR / 'pytest_colab.txt').write_text(test_log, encoding='utf-8')
test_result.check_returncode()

## Validation-only CUDA smoke runs
Run status markers distinguish complete and interrupted directories. Completed runs are skipped. Smoke mode uses at most two epochs and does not evaluate the test split.

In [ ]:
from src.colab_workflow import inspect_run_completion

gru_status = inspect_run_completion(GRU_SMOKE_DIR, 'gru')
print(gru_status)
if not gru_status['complete']:
    subprocess.run([
        sys.executable, 'scripts/run_baselines.py', 'gru',
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(GRU_SMOKE_DIR),
        '--exclude-dynamic-features', 'bis_error',
        '--smoke', '--seed', '42', '--device', 'cuda',
    ], check=True)
subprocess.run([
    sys.executable, 'scripts/verify_colab_smoke.py',
    '--run-dir', str(GRU_SMOKE_DIR), '--dataset-dir', str(DATASET_DIR),
    '--model', 'gru', '--output', str(GRU_SMOKE_DIR / 'smoke_audit.json'),
], check=True)

In [ ]:
attention_status = inspect_run_completion(ATTENTION_SMOKE_DIR, 'attention')
print(attention_status)
if not attention_status['complete']:
    subprocess.run([
        sys.executable, 'scripts/run_attention.py',
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(ATTENTION_SMOKE_DIR),
        '--exclude-dynamic-features', 'bis_error',
        '--smoke', '--seed', '42', '--device', 'cuda',
    ], check=True)
subprocess.run([
    sys.executable, 'scripts/verify_colab_smoke.py',
    '--run-dir', str(ATTENTION_SMOKE_DIR), '--dataset-dir', str(DATASET_DIR),
    '--model', 'attention',
    '--output', str(ATTENTION_SMOKE_DIR / 'smoke_audit.json'),
], check=True)

In [ ]:
subprocess.run([
    sys.executable, 'scripts/benchmark_colab_gpu.py',
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(BENCHMARK_DIR),
    '--batch-sizes', '256,512,1024,2048',
    '--measured-batches', '20', '--warmup-batches', '3', '--seed', '42',
], check=True)

## Interpretation and backend comparability
CPU and CUDA predictions need not be bitwise identical. Compare rows, feature order, architecture, seed configuration, finite outputs, and repeated evaluation of the same checkpoint. Every future paired scientific comparison must use one device/backend for every model and seed. Do not mix CPU and GPU seeds unless the analysis is explicitly designed as backend sensitivity. The existing test set is a development test set, not a pristine final holdout. Future candidate selection remains validation-only; final claims should use previously unseen cases or another pre-specified evaluation design.

In [ ]:
print('Persistent outputs:')
print('Environment:', ENV_OUTPUT_DIR)
print('GRU smoke:', GRU_SMOKE_DIR)
print('Attention smoke:', ATTENTION_SMOKE_DIR)
print('GPU benchmark:', BENCHMARK_DIR)
print('Git commit used:', commit)